In [ ]:
import os
import sys
# add path to custom functions
module_path = os.path.abspath(os.path.join('..'))
if module_path not in sys.path:
    sys.path.append(module_path+"/scripts/py_functions")
# import custom functions
from map_plot_tools import *
from line_plot_tools import *
from colorbar_funcs import *
from data_funcs import *
from domain_funcs import nam_domain_outline, core_site_boxes
# iCESM loading/derivation and significance testing live in scripts/py_functions/ so this
# notebook and LIG127k_analyses_PALEOCALADJUSTED.ipynb run the identical code
from icesm_funcs import (DAT_META, assign_cesm_month_year, derive_dat,
                         monthly_climatology, seasonal_means, windSpd)
from stats_funcs import sigtest, sigtest2n, mask_insignificant

import xarray as xr
xr.set_options(keep_attrs=True)
import numpy as np
#np.set_printoptions(threshold=np.inf) # disable truncation
import pandas as pd 

import cartopy.crs as ccrs
import cartopy.feature as cfeature
from cartopy.mpl.gridliner import LONGITUDE_FORMATTER, LATITUDE_FORMATTER
from shapely.geometry.polygon import LinearRing

import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm, ListedColormap, LinearSegmentedColormap
from matplotlib import cm
import cmocean.cm as cmo
# settings
%config InlineBackend.figure_format = 'retina'

# top level data directory (override with the WORK_DATA_DIR env var; see config/paths.env.example)
dpath0=os.environ.get('WORK_DATA_DIR', '/glade/work/dervlamk')
# save figs here, under a subdir named for this notebook. HPC has no path to OneDrive, so this
# is a separate directory from the laptop-side FIG_OUTPUT_DIR (config/paths.env.example,
# proxy-side only) -- sync the two by hand (rsync/scp) when picking work back up on the laptop.
# Override the root with HPC_FIG_OUTPUT_DIR; defaults to a figures/ subdir next to the PI/,
# LGM/, obs_data/ dirs under WORK_DATA_DIR.
opath=os.path.join(os.environ.get('HPC_FIG_OUTPUT_DIR', f'{dpath0}/nam-interglacial-dD/notebooks/outputs'))
os.makedirs(opath, exist_ok=True)

In [ ]:
# --- LOAD COMPARISON DATA --- #

# IMERG precipitation
# BASELINE: 2001-2018 everywhere in this project (swna_modern_climatology.ipynb uses the same).
#
# Both derived products are built OUTSIDE this notebook, by scripts/nco/make_imerg_climo.sh.
# Deliberately: the source is 216 monthly records on a 0.1 deg global grid, 5.6 GB in memory as
# float32, and deriving the climatology here with groupby/sort/write materialises that several
# times over and kills the kernel. NCO streams the record dimension instead -- about a minute and
# a few hundred MB. Don't move this back into the notebook.
#
# The axis is VALIDATED rather than trusted. The cache written before 2026-08-10 held correct data
# on a mislabelled longitude axis: an older code path replaced lon with np.linspace(0, 360, nlon)
# and rolled by nlon (a no-op), leaving every label 180 deg from the truth. Nothing errored -- the
# IMERG line in LIG127k_analyses_PALEOCALADJUSTED.ipynb was simply averaging the wrong side of
# the planet. Both notebooks read this one cache, so both validate it: a correctly wrapped axis
# stops one cell short of 360, the broken one included both 0 and 360.
obs_dir = f'{dpath0}/obs_data'
imerg_climo_filen = f'{obs_dir}/imerg.gn.2001-2018.climo.nc'
imerg_swna_filen  = f'{obs_dir}/imerg.gn.2001-2018.swna.tseries.nc'

_build_msg = 'run scripts/nco/make_imerg_climo.sh (needs `module load nco`) and re-run this cell'
for f in (imerg_climo_filen, imerg_swna_filen):
    if not os.path.exists(f):
        raise FileNotFoundError(f'{f} not found -- {_build_msg}')

imerg = xr.open_dataset(imerg_climo_filen).precipitation   # 12-month climatology, global, mm/day
_lon  = imerg['lon'].values.astype('float64')
_step = 360 / imerg.sizes['lon']
if not (_lon.max() < 360 and np.allclose(np.diff(_lon), _step, atol=0.05*_step)):
    raise ValueError(f'{imerg_climo_filen}: longitude axis is not a clean 0:360 wrap '
                     f'(max {_lon.max():.4f}, step {np.median(np.diff(_lon)):.8f}, expected '
                     f'{_step:.8f}) -- delete it and {_build_msg}')

# ETOPO05 topography
filen = f'{dpath0}/obs_data/obs.etopo5.zsurf.nc'
etopo_full = xr.open_dataset(f'{filen}').ROSE
etopoSWNA = etopo_full.sel(ETOPO05_X=slice(235,275), ETOPO05_Y=slice(10,42))

# Proxy timeslice mean values
proxydD = pd.read_csv('../data/processed/timeslice_mean_proxy_dDp.csv')

In [ ]:
### +++ SET FILE PATH INFO FOR iCESM1.2 OUTPUT +++ ###

# Points at the per-year (years 801-900) subset produced by
# scripts/nco/subset_tseries.sh + scripts/ncl/pressureRegrid_tseries.ncl, which write into this
# repo's own data/raw/ and data/interim/ (not $WORK_DATA_DIR) -- see data/README.md.
# Both files sit flat in data/raw/ and data/interim/ (no per-case subdirectory),
# named '{varn}.{tag}.0801-0900.tseries.nc' (raw, hybrid-sigma levels) and
# '{varn}.{tag}.0801-0900.tseries.plev.nc' (interim, pressure-level regrid).
# Both the monthly climatology (`dat_climo`, below) and the per-year sample used for significance testing
# (`dat_ts`, below) are derived from this same source, for both cases, so PI and LGM use the same 801-900 window.

raw_varns = ['PRECRC_H2Or', 'PRECRL_H2OR', 'PRECSC_H2Os', 'PRECSL_H2OS', 'PRECRC_HDOr', 'PRECRL_HDOR', 'PRECSC_HDOs', 'PRECSL_HDOS',
             'PRECRC_H216Or', 'PRECRL_H216OR', 'PRECSC_H216Os', 'PRECSL_H216OS', 'PRECRC_H218Or', 'PRECRL_H218OR', 'PRECSC_H218Os', 'PRECSL_H218OS',
             'PRECC', 'PRECL','TS','U','V','OMEGA','Q','Z3','PSL']

files = {}
for case, tag in [('pi', 'iPI.01'), ('lgm', 'i21ka.03')]:
    files[case] = {}
    raw_dir = f'{module_path}/data/raw'
    interim_dir = f'{module_path}/data/interim'
    for varn in raw_varns:   # PS is regrid-only (used by the NCL step), never loaded here
        files[case][varn] = f'{raw_dir}/{varn}.{tag}.0801-0900.tseries.nc'
    for varn in ['U', 'V', 'OMEGA', 'Q', 'Z3']:
        files[case][varn] = f'{interim_dir}/{varn}.{tag}.0801-0900.tseries.plev.nc'   # overrides the hybrid-level path above


In [ ]:
### +++ PROCESS iCESM1.2 OUTPUT +++ ###

cases=['pi', 'lgm']

#== load per-year raw variables (years 801-900, 1200 monthly records)
# assign_cesm_month_year() shifts the time axis back two days and attaches `month`/`year`
# coords: CESM h0 timestamps the END of each averaging period (Jan's mean is stamped
# 0801-02-01), so a raw .dt.month/.dt.year read mislabels every record by one month and
# misassigns December into the next year. Everything downstream reads those two coords, never
# the timestamps -- see scripts/py_functions/icesm_funcs.py.
raw = {case: {} for case in cases}
for case in cases:
    for varn in raw_varns:
        time_coder = xr.coders.CFDatetimeCoder(use_cftime=True)
        da = xr.open_dataset(files[case][varn], decode_times=time_coder)[varn]
        raw[case][varn] = assign_cesm_month_year(da)


#== construct processed data dictionaries
dat_varns = ['TS', 'OMEGA', 'U', 'V', 'PSL', 'Q', 'Z3', 'PRECC', 'PRECL', 'PRECT', 'dDp', 'd18Op']

# per-year derived data, for significance testing (see the significance-testing section below).
# dDp/d18Op come back as plain per-mil ratios -- the precipitation weighting is applied by
# seasonal_means() below, where the season being averaged is known.
dat_ts = derive_dat(raw, cases, dat_varns)

# monthly climatology
raw_clim = monthly_climatology(raw, cases, raw_varns)
dat_climo = derive_dat(raw_clim, cases, dat_varns)
print('Calculated monthly climatologies.')

# seasonal and annual averaging. PRECT-weighted for dDp/d18Op, plain means for everything else.
seasons = ['ann', 'jas'] 
seas_mean, ann_seas_mean = seasonal_means(dat_ts, cases, dat_varns, seasons)
print('Calculated annual and seasonal averages.')


#== iCESM1.2 topography
in_file = f'{module_path}/data/raw/topo_21ka_remap_19x25.mod.170428.sm9.nc'
LANDFR = xr.open_dataset(in_file).LANDFRAC
PHIS = xr.open_dataset(in_file).PHIS
# the available variable is "surface geopotential" in units m2/s2
# approximate the surface geometric elevation by dividing by gravitational acceleration
g = 9.80665 # m/s2
zsurf = PHIS / g
zsurf.attrs['units'] = 'm'
zsurf.attrs['long_name'] = 'surface elevation'
zsurf.attrs['source_file'] = '/glade/work/jiangzhu/data/inputdata/cesm120ka_ICEG6/21ka/topo_21ka_remap_19x25.mod.170428.sm9.nc'

## Statistical significance of LGM$-$PI differences

The cell below runs `sigtest2n()` (Welch's, unpaired -- `pi` and `lgm` are independent
simulations, not paired samples) to get `lgm_pi_diff`/`lgm_pi_diff_mask`/`lgm_pi_ptvals` for
both the 2D surface fields and the 3D pressure-level fields.

In [ ]:
### +++ LGM-PI SIGNIFICANCE TESTING +++ ###

lgm_pi_diff = {varn: {} for varn in dat_varns}
lgm_pi_diff_mask = {varn: {} for varn in dat_varns}
lgm_pi_ptvals = {varn: {} for varn in dat_varns}

print('Significance testing LGM vs PI for:')
for varn in dat_varns:
    print(f'...{varn}')
    for season in seasons:
        diff_, diff_mask_, ptvals_ = sigtest2n(
            ann_seas_mean['lgm'][varn][season], ann_seas_mean['pi'][varn][season],
            seas_mean['lgm'][varn][season], seas_mean['pi'][varn][season]
        )
        lgm_pi_diff[varn][season] = diff_
        lgm_pi_diff_mask[varn][season] = diff_mask_
        lgm_pi_ptvals[varn][season] = ptvals_

print('Done.')

## Core-site box means (LGM$-$PI)

In [ ]:
# --- CORE-SITE MEAN LGM-PI dD_precip --- #
        
# The boxes are defined once, in scripts/py_functions/domain_funcs.py -> core_site_boxes(), and
# are the same object drawn on the maps below (ax.add_geometries(...)) -- so the region shown
# and the region averaged here cannot drift apart. Same box definitions and same cos(lat)
# weighted-mean approach as swna_modern_climatology.ipynb, which averages the OIPC isoscape over
# these boxes. Unlike OIPC (a terrestrial-only isoscape), the model grid has no ocean mask to
# worry about, so this is a plain weighted mean over the box, no NaN-skipping needed.
#
# The model grid is 0:360 in longitude; core_site_boxes() is -180:180 (matching the proxy lon/
# lat columns and how boxes are drawn on these PlateCarree maps), so bounds are converted with
# `% 360` before slicing.

proxy_varns = ['dDp']

# Which proxy core sits in which model box. Stated explicitly rather than left to the row order
# of the CSV lining up with the insertion order of core_site_boxes() -- both happen to be
# (Guaymas/DSDP-480-479, Mazatlan/NH22P) today, and nothing enforced it. Reordering either one
# would have silently swapped the two sites' proxy values.
SITE_TO_CORE = {'Guaymas': 'DSDP_480_479', 'Mazatlan': 'NH22P'}

site_boxes = core_site_boxes()
site_diff  = {site: { varn:{} for varn in proxy_varns } for site in site_boxes}

proxy_by_core = proxydD.set_index('core_name')
missing = set(SITE_TO_CORE.values()) - set(proxy_by_core.index)
if missing:
    raise KeyError(f'core name(s) {sorted(missing)} not in the proxy CSV -- '
                   f'it has {sorted(proxy_by_core.index)}')

for site, poly in site_boxes.items():
    w, s, e, n = poly.bounds  # shapely: (min_lon, min_lat, max_lon, max_lat)
    for varn in proxy_varns:
        for season in seasons:
            box = lgm_pi_diff[varn][season].sel(lon=slice(w % 360, e % 360), lat=slice(s, n))
            weights = np.cos(np.deg2rad(box.lat))
            site_diff[site][varn][season] = float(box.weighted(weights).mean(('lat', 'lon')))

for varn in proxy_varns:
    print(f'Model LGM-PI {varn} [per mil] -- cos(lat)-weighted mean over the averaging box around each core site')
    print(f"{'season':>8}" + ''.join(f'{site:>12}' for site in site_boxes))
    for season in seasons:
        print(f'{season:>8}' + ''.join(f"{site_diff[site][varn][season]:>12.2f}" for site in site_boxes))


# Calculate LGM-Late Holocene dDprecip differences for the proxy records and print summary.
# Built in site_boxes order via SITE_TO_CORE, so it stays aligned with core_lons/core_lats and
# with the map markers in the figures below, which index it positionally.
core_dDdiff = np.array([proxy_by_core.loc[SITE_TO_CORE[site], 'lgm_dD']
                        - proxy_by_core.loc[SITE_TO_CORE[site], 'late_holocene_dD']
                        for site in site_boxes])

print('\nProxy LGM-LH dDp [per mil] -- at each specific core site')
print(f"{'':>8}" + ''.join(f'{site:>12}' for site in site_boxes))
print(f"{'':>8}" + ''.join(f'{value:>12.2f}' for value in core_dDdiff))

### Per-year box means and interannual spread

The cell above collapses the 100 simulated years before averaging over each box, so its box
means carry no measure of how variable $\delta D_p$ is from year to year. This cell rebuilds the
same box means from `ann_seas_mean` instead, one value per simulation year, which is what gives
the model an error bar to set against the proxy's.

In [ ]:
# --- PER-YEAR CORE-SITE BOX MEANS, AND THE LGM-PI ANOMALY WITH ITS INTERANNUAL SPREAD --- #

# Identical slicing and cos(lat) weighting to the cell above -- the one difference is the input:
# ann_seas_mean[case]['dDp'][season] carries a `year` dimension (100 simulated years) where
# lgm_pi_diff has already collapsed it. No float() cast, so the year axis survives.

site_years = {site: {case: {} for case in cases} for site in site_boxes}
for site, poly in site_boxes.items():
    w, s, e, n = poly.bounds
    for case in cases:
        for season in seasons:
            box = ann_seas_mean[case]['dDp'][season].sel(lon=slice(w % 360, e % 360),
                                                         lat=slice(s, n))
            weights = np.cos(np.deg2rad(box.lat))
            site_years[site][case][season] = box.weighted(weights).mean(('lat', 'lon'))

# LGM-PI anomaly and its 1-sigma interannual uncertainty, per site and season.
#
# sigma is the quadrature sum of each case's own spread across years, NOT the spread of a
# per-year difference: `pi` and `lgm` are independent 100-year simulations, so there is no
# year-to-year pairing between them to difference. ddof=1 because the 100 years are a sample of
# each climate, not the whole of it -- the same convention as the interannual error bars in
# LIG127k_analyses_PALEOCALADJUSTED.ipynb.
site_anom = {site: {} for site in site_boxes}
for site in site_boxes:
    for season in seasons:
        pi_yr, lgm_yr = site_years[site]['pi'][season], site_years[site]['lgm'][season]
        delta = float(lgm_yr.mean() - pi_yr.mean())
        sd    = float(np.hypot(float(pi_yr.std(ddof=1)), float(lgm_yr.std(ddof=1))))

        # The mean of the per-year means must equal the collapsed difference computed above.
        # It does because seasonal_means() weights months by precipitation WITHIN each year and
        # then averages years equally, so both paths apply the same weights (see icesm_funcs.py).
        # Pooling the weighting across the whole record instead would break this by ~0.2 permil
        # -- small, but it would mean the maps plotted one number while this figure plotted
        # another. This assertion is what catches that if the weighting is ever changed back.
        assert abs(delta - site_diff[site]['dDp'][season]) < 1e-3, (
            f'{site}/{season}: per-year mean {delta:.6f} != collapsed difference '
            f"{site_diff[site]['dDp'][season]:.6f} -- has the weighting in "
            'icesm_funcs.seasonal_means() changed?')

        site_anom[site][season] = {'delta': delta, 'sd': sd,
                                   'sd_pi': float(pi_yr.std(ddof=1)),
                                   'sd_lgm': float(lgm_yr.std(ddof=1))}

print('Model LGM-PI dDp [per mil] -- box mean, with 1 sigma across the 100 simulated years')
print(f"{'season':>8}" + ''.join(f'{site:>22}' for site in site_boxes))
for season in seasons:
    row = ''.join(f"{site_anom[site][season]['delta']:>13.2f} +/-{site_anom[site][season]['sd']:>6.2f}"
                  for site in site_boxes)
    print(f'{season:>8}' + row)

print('\nper-case interannual spread [per mil, 1 sigma, ddof=1]')
print(f"{'season':>8}" + ''.join(f'{site:>22}' for site in site_boxes))
for season in seasons:
    row = ''.join(f"   PI{site_anom[site][season]['sd_pi']:>7.2f}  LGM{site_anom[site][season]['sd_lgm']:>7.2f}"
                  for site in site_boxes)
    print(f'{season:>8}' + row)

# FIGS

In [ ]:
### +++ USER-DEFINED FIG SETTINGS AND INPUTS +++ ###

# season plotted by the figures below -- select any key in `seasons` (set in the processing cell above)
season = 'jas'

# levels pulled from the 3D pressure-level fields
omega_lev = 500.0   # mb, vertical velocity panel
wind_lev  = 850.0   # mb, wind vectors

# plot specs
bbox     ={'boxstyle':'square','fc':'white','ec':'black','alpha':1,'pad':0.2}
text_kw  ={'color':'k', 'weight':'bold', 'size':14, 'ha':'center', 'va':'bottom'}
text_kw1 ={'color':'k', 'weight':'bold', 'size':18, 'ha':'center', 'va':'bottom'}
letters = np.array(['a','b','c'])

# map specs
trans = ccrs.PlateCarree()
proj  = ccrs.PlateCarree()
map_bnds = [-120., -82.5, 10., 36.]

# NAM domain outline, shared with swna_modern_climatology.ipynb (scripts/py_functions/
# domain_funcs.py). site_boxes comes from the core-site box means cell above -- same object,
# so what's drawn here and what's averaged there can't drift apart.
nam_domain = nam_domain_outline()

# vector specs
skip_n=1
w=0.0075
scalef=10
key_length=2

# Lat/Lon vars.
# Built in site_boxes order via SITE_TO_CORE, the same way core_dDdiff is -- the figures below
# zip the three together positionally (core_lons[j], core_lats[j], core_dDdiff[j]), so all
# three have to be ordered the same way. Taking these two straight off the CSV instead would
# put them in row order, which agrees today but is not enforced by anything.
core_lons = np.array([proxy_by_core.loc[SITE_TO_CORE[site], 'lon'] for site in site_boxes])
core_lats = np.array([proxy_by_core.loc[SITE_TO_CORE[site], 'lat'] for site in site_boxes])
model_lons, model_lats = lgm_pi_diff['PRECT'][season].lon, lgm_pi_diff['PRECT'][season].lat

# construct dictionaries of per-panel model data + color specs
panels = [
    dict(title=r'$\mathbf{\Delta}$ Precipitation',                 # ΔPrecipitation
         data=lgm_pi_diff_mask['PRECT'][season],
         cmap=get_settings(field='precip', diff=True)[0], vmin=-2.5, vmax=2.5, nlevels=21,
         cbar_ticks=[-2, -1, 0, 1, 2], cbar_label='[mm day$^{-1}$]',
         proxy_colored=False, # True to shade core sites on color map
         quiver=False         # True to draw wind vectors
        ),
    # +/-12 permil in 1 permil steps. Scaled to the field as it actually is once dDp is
    # correctly precipitation-weighted (seasonal_means(), not derive_dat()): the masked JAS
    # difference runs -17.5 to +12.1 over this extent, with 90% of cells inside +/-13, so
    # extend='both' absorbs the tail rather than the scale being stretched to fit it. The old
    # +/-3 was sized for the field before the weighting fix, when it was ~a tenth of its true
    # size. The proxy markers (-7.7, +5.3 permil) share this norm and land mid-scale.
    dict(title=r'$\mathbf{\Delta}\ \mathbf{\delta D_{precip}}$',   # ΔδD_precip
         data=lgm_pi_diff_mask['dDp'][season],
         cmap=cm.RdBu_r, vmin=-12, vmax=12, nlevels=25,
         cbar_ticks=[-12, -9, -6, -3, 0, 3, 6, 9, 12], cbar_label=u'[‰]',
         proxy_colored=True,  # True to shade core site markers on cmap
         quiver=False         # True to draw wind vectors
        ),
    dict(title=rf'$\mathbf{{\Delta}}\ \mathbf{{\omega_{{{omega_lev:.0f}}}}}$ and {wind_lev:.0f} mb Wind',
         data=lgm_pi_diff_mask['OMEGA'][season].sel(lev_p=omega_lev),
         cmap=cmo.balance, vmin=-0.05, vmax=0.05, nlevels=21,
         cbar_ticks=[-0.04, -0.02, 0, 0.02, 0.04], cbar_label='[Pa s$^{-1}$]',
         proxy_colored=False, # True to shade core site markers on cmap
         quiver=True          # True to draw wind vectors
        ),
]
# colormap normalization
for p in panels:
    p['norm'] = mpl.colors.BoundaryNorm(np.linspace(p['vmin'], p['vmax'], p['nlevels']), p['cmap'].N)

In [ ]:
### +++ LGM-PI CLIMATOLOGY DIFFERENCES FIGURE +++ ###

fig, ax = plt.subplots(nrows=1, ncols=3,
                       figsize=(16,4.5),
                       subplot_kw={'projection': proj},
                       layout='constrained')

# figure title
fig.text(.5,1,' iCESM1.2 LGM (21ka) $-$ PI differences : '+season, **text_kw)

for i, (axi, p) in enumerate(zip(ax, panels)):
    
    # add sub-panel title and label
    axi.text(map_bnds[0]-((map_bnds[0]-map_bnds[1])/2), map_bnds[3]+0.1, p['title'], **text_kw)
    axi.text(map_bnds[0]+1, map_bnds[3]+0.5, letters[i], **text_kw1)
    
    # plot field
    p['cf'] = axi.pcolormesh(model_lons, model_lats, p['data'], cmap=p['cmap'], norm=p['norm'], transform=trans)
    
    # plot vectors
    if p['quiver']:
        # masked, only significant wind changes get an arrow
        u_lev = lgm_pi_diff_mask['U'][season].sel(lev_p=wind_lev)
        v_lev = lgm_pi_diff_mask['V'][season].sel(lev_p=wind_lev)
        q1 = axi.quiver(u_lev.lon[::skip_n], u_lev.lat[::skip_n], u_lev[::skip_n,::skip_n], v_lev[::skip_n,::skip_n],
                         color='k', width=w, scale=scalef, scale_units='inches', units='height',
                         transform=trans, zorder=100)
        axi.quiverkey(q1, .95, 1.035, key_length, rf'{key_length} m/s',
                      labelcolor='k', labelpos='W', fontproperties={'size':9})
        
    # plot core site markers
    if p['proxy_colored']:
        for j, site_data in enumerate(core_dDdiff):
            axi.scatter(x=core_lons[j], y=core_lats[j], c=site_data,
                        cmap=p['cmap'], norm=p['norm'], alpha=1, ec='k', s=150,
                        transform=trans, zorder=100)
            axi.text(core_lons[j]-1.1, core_lats[j], f'{site_data:.1f}‰',
                     fontsize=10, weight='bold', ha='right', bbox=bbox, zorder=100)
        # add averaging area boxes around core sites
        axi.add_geometries(list(site_boxes.values()), crs=trans, fc='none', ec='k', lw=1,
                            linestyle='--', zorder=9)
    else:
        axi.scatter(core_lons, core_lats, ec='k', fc='k', s=150, alpha=1, transform=trans, zorder=100)

    #== map formatting common to all three panels
    # model topography
    axi.contour(zsurf.lon, zsurf.lat, zsurf, levels=np.linspace(500,4000,11), linewidths=0.5, colors='k', transform=trans)
    #axi.contour(zsurf.lon, zsurf.lat, zsurf, levels=[1], linewidths=1.2, colors='k', transform=trans)
    axi.coastlines(lw=1)
    # NAM domain polygon outline
    axi.add_geometries([nam_domain], crs=trans, fc='none', ec='r', lw=2, linestyle='--', zorder=9)
    # grid lines
    axi.set_extent(map_bnds, crs=trans)
    gl = axi.gridlines(crs=trans, lw=0.5, colors='black', alpha=1.0, linestyle='--', zorder=10, draw_labels=True)
    gl.top_labels = False; gl.right_labels = False; gl.left_labels = (i == 0)

# colorbars
for i, p in enumerate(panels):
    cax = fig.add_axes([0.05 + i*0.325, 0, 0.275, 0.05])
    cbar = fig.colorbar(p['cf'], ticks=p['cbar_ticks'], orientation='horizontal', extend='both', cax=cax)
    cbar.set_label(p['cbar_label'], weight='normal', labelpad=5, rotation=0)
    cbar.ax.tick_params(labelsize=10)
    for tick in cbar.ax.xaxis.get_major_ticks():
        tick.label1.set_fontweight('normal')

# save output
#plt.savefig(os.path.join(opath, "LGM-PI_icesm1p2_diffs.png"), dpi=1200, bbox_inches='tight')

## Convective vs. Large-Scale Precip

In [ ]:
### +++ LGM-PI CONVECTIVE vs. LARGE-SCALE PRECIP DIFFERNECES FIGURE +++ ###

# --- Plot-specific settings --- #
# Map/text/domain specs are deliberately NOT redefined here -- they are reused from the fig
# settings cell so the two figures cannot drift apart.

# both panels share one cmap/norm and one colorbar
pc_cmap, _, _, _ = get_settings(field='precip', diff=True)
pc_vmin = -2.5
pc_vmax = 2.5
pc_levels = 21
pc_norm = mpl.colors.BoundaryNorm(np.linspace(pc_vmin, pc_vmax, pc_levels), pc_cmap.N)

pc_panels = [
    dict(title='PRECC  (convective)',   data=lgm_pi_diff_mask['PRECC'][season]),
    dict(title='PRECL  (large-scale)',  data=lgm_pi_diff_mask['PRECL'][season]),
]

# --- Make plot --- #
fig, ax = plt.subplots(nrows=1, ncols=2,
                       figsize=(11,4.5),
                       subplot_kw={'projection': proj},
                       layout='constrained')

# figure title
fig.text(.5, 1, f' iCESM1.2 LGM (21ka) $-$ PI precipitation partition : {season}', **text_kw)

for i, (axi, p) in enumerate(zip(ax, pc_panels)):

    # add sub-panel title and label
    axi.text(map_bnds[0]-((map_bnds[0]-map_bnds[1])/2), map_bnds[3]+0.1, p['title'], **text_kw)
    axi.text(map_bnds[0]+1, map_bnds[3]+0.5, letters[i], **text_kw1)

    # plot field
    p['cf'] = axi.pcolormesh(model_lons, model_lats, p['data'],
                             cmap=pc_cmap, norm=pc_norm, transform=trans)

    # core site markers
    axi.scatter(core_lons, core_lats, ec='k', fc='k', s=150, alpha=1, transform=trans, zorder=100)

    #== map formatting common to both panels
    # model topography
    axi.contour(zsurf.lon, zsurf.lat, zsurf, levels=np.linspace(500,4000,11), linewidths=0.5, colors='k', transform=trans)
    axi.coastlines(lw=1)
    # NAM domain polygon outline
    axi.add_geometries([nam_domain], crs=trans, fc='none', ec='r', lw=2, linestyle='--', zorder=9)
    # grid lines
    axi.set_extent(map_bnds, crs=trans)
    gl = axi.gridlines(crs=trans, lw=0.5, colors='black', alpha=1.0, linestyle='--', zorder=10, draw_labels=True)
    gl.top_labels = False; gl.right_labels = False; gl.left_labels = (i == 0)

# one shared colorbar -- both panels use the same cmap/norm
cax = fig.add_axes([0.15, -.05, 0.7, 0.05])
cbar = fig.colorbar(pc_panels[-1]['cf'], ticks=[-2, -1, 0, 1, 2],
                    orientation='horizontal', extend='both', cax=cax)
cbar.set_label(r'$\Delta$ Precipitation [mm day$^{-1}$]', weight='normal', labelpad=5, rotation=0)
cbar.ax.tick_params(labelsize=10)

# save output
plt.savefig(os.path.join(opath, f'LGM-PI_icesm1p2_diffs_precc_precl.png'), dpi=1200, bbox_inches='tight')

## Moisture Transport

In [ ]:
#=== Calculate moisture transport

# built from the monthly climatology (dat_climo), not the per-year record -- the maps this
# feeds are climatological. `dat` was the pre-timeseries name for this dict and no longer exists.
qv = { 'pi':{}, 'lgm':{} }
qu = { 'pi':{}, 'lgm':{} }
mt = { 'pi':{}, 'lgm':{} }

for key in ['pi','lgm']:
    qu[key] = dat_climo[key]['U']*dat_climo[key]['Q']
    qv[key] = dat_climo[key]['V']*dat_climo[key]['Q']
    mt[key] = windSpd(qu[key], qv[key]) 

## Model vs. proxy $\Delta\delta D_p$

Sets the model's LGM$-$PI box-mean anomaly against the measured LGM$-$late-Holocene anomaly at
each core, each with its own uncertainty. The two error bars are **not** the same quantity and
the figure says so in-axes: the model's is interannual spread across the 100 simulated years,
the proxy's are the spread of samples inside the window and the Monte Carlo uncertainty on the
window mean.

In [ ]:
# --- PROXY LGM - LATE HOLOCENE ANOMALY, WITH BOTH UNCERTAINTIES --- #

# Reference window is late_holocene (0-4 ka), NOT holocene (0-11.7 ka). Every published anomaly
# in this project is against 0-4 ka, and swapping the two flips the sign of the DSDP-480/479 LIG
# anomaly -- see data/processed/README.md.
PROXY_REF = 'late_holocene'
PROXY_INT = 'lgm'
ANOM = f'{PROXY_INT}_minus_{PROXY_REF}_dD'

# The anomaly columns are written by the proxy-side notebook
# (dDwax_timeslice-means_timeseries.ipynb), which runs on the laptop only -- the .mat ensembles
# it needs never come to Casper, only the CSV does. So this cell works either way: with the
# exported columns when the CSV has made the round trip, and from the timeslice means directly
# when it has not. The model half of the figure is identical under both paths; only the proxy
# whiskers differ, and the fallback simply has no Monte Carlo one to draw.
proxy_anom = {}
have_export = ANOM in proxydD.columns

for site in site_boxes:
    r = proxy_by_core.loc[SITE_TO_CORE[site]]
    ref_sd = float(r[f'{PROXY_REF}_dD_stddev'])
    if have_export:
        entry = dict(delta=float(r[ANOM]),
                     sd_samples=float(r[f'{ANOM}_stddev_samples']),
                     sd_mc=float(r[f'{ANOM}_stddev_mc']))
    else:
        # mean difference, and the two window spreads added in quadrature -- the same
        # definition the exported _stddev_samples column carries, computed here instead
        entry = dict(delta=float(r[f'{PROXY_INT}_dD'] - r[f'{PROXY_REF}_dD']),
                     sd_samples=float(np.hypot(r[f'{PROXY_INT}_dD_stddev'], ref_sd)),
                     sd_mc=np.nan)
    # sample counts, for the annotation below. Absent from the pre-update CSV.
    for key, col in [('n', f'{PROXY_INT}_n'), ('n_ref', f'{PROXY_REF}_n')]:
        entry[key] = int(r[col]) if col in proxydD.columns else None
    # A reference window holding a single sample has no spread of its own, so it contributes
    # nothing to sd_samples and the whisker understates the real uncertainty. Flagged on the
    # figure. Without the _n columns a zero stddev is the same signal -- it is what one sample
    # produces -- so the flag works on the fallback path too, where NH22P's 0-4 ka window is
    # a single measurement.
    entry['ref_degenerate'] = (entry['n_ref'] == 1) if entry['n_ref'] is not None else (ref_sd == 0)
    proxy_anom[site] = entry

print(f"Proxy anomaly source: {'exported columns' if have_export else 'FALLBACK'} "
      f"({'' if have_export else 'no '}{ANOM} in the CSV)")
if not have_export:
    print('  -> Monte Carlo whisker unavailable; re-run fig3 on the laptop, commit the CSV, and pull.')
print(f'\nProxy {PROXY_INT.upper()} - {PROXY_REF.replace("_", " ")} dDp [per mil]')
for site in site_boxes:
    a = proxy_anom[site]
    mc = 'n/a' if np.isnan(a['sd_mc']) else f"{a['sd_mc']:.2f}"
    n = '' if a['n'] is None else f"   (n = {a['n']} vs {a['n_ref']})"
    flag = '   [reference window is a single sample]' if a['ref_degenerate'] else ''
    print(f"{site:>10} ({SITE_TO_CORE[site]:>12}): {a['delta']:>7.2f}"
          f"   sd_samples {a['sd_samples']:>5.2f}   sd_mc {mc:>5}{n}{flag}")

In [ ]:
### +++ MODEL vs. PROXY dD_precip ANOMALY FIGURE +++ ###

# Anomaly only -- one panel, no absolute values. Both quantities are differences against their
# own reference state (PI for the model, the 0-4 ka window for the proxy), and their absolute
# levels are not comparable: iCESM dDp is a grid-box mean of precipitation, the proxy is
# leaf-wax-derived dDp at a point. The anomalies are what the comparison is about.
#
# `season` comes from the fig settings cell above, so this figure and the maps are always the
# same season.

# two series, so a legend is always drawn; identity is never carried by color alone (the two
# also differ in marker and in whisker structure). Colors are CVD-checked: worst-pair
# separation dE 19.7 under protanopia, 26.4 normal vision.
col_model = '#2A6FB0'
col_proxy = '#C1622F'

site_labels = {'Guaymas':  'Guaymas Basin\n(DSDP-480/479)',
               'Mazatlan': 'Mazatlán Margin\n(NH22P)'}

x = np.arange(len(site_boxes))
dx = 0.12

fig, ax = plt.subplots(nrows=1, ncols=1, figsize=(6.5, 5.5), layout='constrained')

ax.text(0.5, 1.01, 'LGM $\\Delta\\ \\delta D_{precip}$: MODEL vs. PROXY',
        transform=ax.transAxes, **text_kw)

# zero line first, so the markers sit on top of it
ax.axhline(0, lw=0.8, c='0.5', zorder=1)

for i, site in enumerate(site_boxes):
    m = site_anom[site][season]
    p = proxy_anom[site]

    # --- model: one whisker, interannual spread across the 100 simulated years
    ax.errorbar(x[i] - dx, m['delta'], yerr=m['sd'],
                marker='o', ms=9, mfc=col_model, mec='w', mew=1.2,
                c=col_model, elinewidth=1.2, capsize=3, capthick=1.2, zorder=3,
                label='iCESM1.2 LGM$-$PI, box mean' if i == 0 else '_Hidden')

    # --- proxy: two overlaid whiskers on one marker. The thin outer one is the spread of the
    # samples inside each window (the analog of the model's interannual spread); the thick
    # inner one is the Monte Carlo uncertainty on the window mean. Drawn thin-first so the
    # short thick bar stays legible on top of the long thin one.
    ax.errorbar(x[i] + dx, p['delta'], yerr=p['sd_samples'],
                marker='none', c=col_proxy, elinewidth=1.0, capsize=3, capthick=1.0, zorder=3,
                label='proxy $\\delta D_p$, LGM$-$late Holocene' if i == 0 else '_Hidden')
    if not np.isnan(p['sd_mc']):
        ax.errorbar(x[i] + dx, p['delta'], yerr=p['sd_mc'],
                    marker='none', c=col_proxy, elinewidth=3.2, capsize=0, zorder=4,
                    label='_Hidden')
    ax.plot(x[i] + dx, p['delta'], marker='D', ms=8, mfc=col_proxy, mec='w', mew=1.2,
            ls='none', zorder=5, label='_Hidden')

    # A one-sample reference window contributes no spread of its own, so the thin whisker is
    # narrower than the real uncertainty and would otherwise read as a confident measurement.
    if p['ref_degenerate']:
        ax.annotate('reference window:\n1 sample',
                    xy=(x[i] + dx, p['delta']), xytext=(x[i] + dx + 0.08, p['delta']),
                    ha='left', va='center', size='small', color='0.35')

ax.set_xticks(x)
ax.set_xticklabels([site_labels[s] for s in site_boxes], size='large')
ax.set_xlim(-0.5, len(site_boxes) - 0.5)
ax.set_ylabel(r'$\Delta\ \delta D_{p}$ [‰]', size='x-large')
ax.tick_params(axis='both', direction='in', labelsize='large')
ax.tick_params(axis='x', length=0)
for side in ('top', 'right'):
    ax.spines[side].set_visible(False)

# What each whisker actually means. The three are different quantities and the figure is
# misleading without this line -- same convention as the interannual note in
# LIG127k_analyses_PALEOCALADJUSTED.ipynb's precip climatology figure.
_nyears = len(site_years[next(iter(site_boxes))]['pi'][season].year)
notes = [f'model: $\\pm1\\sigma$ across {_nyears} simulated years',
         'proxy thin: $\\pm1\\sigma$ of samples in window',
         'proxy thick: $\\pm1\\sigma$ Monte Carlo, window mean']
if np.isnan(proxy_anom[next(iter(site_boxes))]['sd_mc']):
    notes = notes[:2]
ax.text(0.015, 0.02, '\n'.join(notes), transform=ax.transAxes,
        ha='left', va='bottom', size='small', color='0.35')

# Window definitions, stated rather than left to the caption (lower right -- upper
# right is where the Mazatlan whisker and its annotation sit). The model is a 21 ka
# equilibrium snapshot -- a single climate state, not an averaging window like the proxy's.
ax.text(0.985, 0.02, f'{season.upper()}\nmodel: 21 ka vs. PI\nproxy: 18–24 ka vs. 0–4 ka',
        transform=ax.transAxes, ha='right', va='bottom', size='small', color='0.35')

# legend inside the axes, upper left -- the region the data leaves empty. Outside-above would
# push the title off the top of the figure.
ax.legend(loc='upper left', ncols=1, frameon=False, prop={'size': 'medium'})

plt.savefig(os.path.join(opath, 'LGM-PI_dDp_box_vs_proxy.png'), dpi=1200, bbox_inches='tight')